# Unlimited-OCR batch=2 forward debug

这个 notebook 对应服务器脚本 `/mnt1/yixuan/unlimited-ocr-posttrain/debug/debug_batch2_forward.py`，但拆成更细的 cell，方便你在 VS Code/Jupyter 里边跑边看源码调用链。

本 notebook 只解释服务器上的 remote-code：

`/mnt1/yixuan/unlimited-ocr-posttrain/hf_cache/modules/transformers_modules/baidu_Unlimited_hyphen_OCR/`

核心判断：这个 batch2 forward 是两个独立 conversation/sample；`len(batch["images"]) == 2`。普通 `model(..., use_cache=False)` 不会触发 ring-buffer sliding-window decode，它只跑完整序列 forward。sliding-window 主要在 `model.generate()` / `model.infer()` 的 decode 阶段生效。


In [ ]:
# 先跑这一格。改 GPU 后需要 Restart Kernel，避免 CUDA 已经初始化。
import os
from pathlib import Path

GPU = "0"
ROOT = Path("/mnt1/yixuan/unlimited-ocr-posttrain")
MODEL_DIR = ROOT / "models" / "baidu_Unlimited-OCR"
DEBUG_DIR = ROOT / "debug"
HF_MODULE_DIR = ROOT / "hf_cache" / "modules" / "transformers_modules" / "baidu_Unlimited_hyphen_OCR"
MODELING_UOCR = HF_MODULE_DIR / "modeling_unlimitedocr.py"
MODELING_DSV2 = HF_MODULE_DIR / "modeling_deepseekv2.py"

os.environ["CUDA_VISIBLE_DEVICES"] = GPU
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")
os.environ.setdefault("HF_HOME", str(ROOT / "hf_cache"))
os.environ.setdefault("TRANSFORMERS_CACHE", str(ROOT / "hf_cache"))
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")

print("MODEL_DIR:", MODEL_DIR)
print("HF_MODULE_DIR:", HF_MODULE_DIR)
print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"])


## 0. 这次要看的调用链

先把脑子里的路径固定下来。你在 forward cell 里调用的是：

```text
model(...)
  -> UnlimitedOCRForCausalLM.forward              modeling_unlimitedocr.py:617
      -> self.model(...)                          self.model is UnlimitedOCRModel
          -> UnlimitedOCRModel.forward            modeling_unlimitedocr.py:449
              -> image encoder + masked_scatter   modeling_unlimitedocr.py:480-582
              -> super().forward(...)             DeepseekV2Model.forward
                  -> DeepseekV2DecoderLayer.forward
                      -> SlidingWindowLlamaAttention.forward, but normal full forward path if use_cache=False
```

断点建议：

```text
modeling_unlimitedocr.py:617  UnlimitedOCRForCausalLM.forward
modeling_unlimitedocr.py:642  self.model(...) call
modeling_unlimitedocr.py:480  enter image branch
modeling_unlimitedocr.py:493  with torch.no_grad() around visual encoder
modeling_unlimitedocr.py:582  masked_scatter_ image embeddings into text embeddings
modeling_unlimitedocr.py:587  enter DeepseekV2Model.forward
modeling_deepseekv2.py:1749 decoder_layer(...)
modeling_deepseekv2.py:1232 SlidingWindowLlamaAttention class
```


In [ ]:
import importlib
import inspect
import sys
import textwrap
import time

import torch
from IPython.display import display
from PIL import Image
from transformers import AutoModel, AutoTokenizer

if str(DEBUG_DIR) not in sys.path:
    sys.path.insert(0, str(DEBUG_DIR))

from debug_batch2_forward import build_sample, collate_batch, ensure_debug_images

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("visible device:", torch.cuda.get_device_name(0))


In [ ]:
def show_source(path: Path, start: int, end: int) -> None:
    """Print a line-numbered snippet from the server source file."""
    lines = path.read_text().splitlines()
    for lineno in range(start, end + 1):
        print(f"{lineno:5d}: {lines[lineno - 1]}")

print("Unlimited source exists:", MODELING_UOCR.exists(), MODELING_UOCR)
print("DeepSeekV2 source exists:", MODELING_DSV2.exists(), MODELING_DSV2)


## 1. 加载模型，并确认 AutoModel 实际拿到的是 CausalLM

`config.json` 里把 `AutoModel` 映射到了 `modeling_unlimitedocr.UnlimitedOCRForCausalLM`，所以这里虽然写的是 `AutoModel.from_pretrained`，实际对象就是 CausalLM。

这一格会缓存 `tokenizer/model/remote_mod`。后面重复跑不会重新加载权重，除非把 `FORCE_RELOAD=True`。


In [ ]:
FORCE_RELOAD = False

if FORCE_RELOAD:
    for name in ["model", "tokenizer", "remote_mod"]:
        globals().pop(name, None)
    torch.cuda.empty_cache()

if "model" not in globals():
    print("Loading tokenizer/model from", MODEL_DIR)
    t0 = time.time()
    tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), trust_remote_code=True)
    model = AutoModel.from_pretrained(
        str(MODEL_DIR),
        trust_remote_code=True,
        use_safetensors=True,
        torch_dtype=torch.bfloat16,
    ).eval().cuda()
    remote_mod = importlib.import_module(model.__class__.__module__)
    print(f"loaded in {time.time() - t0:.1f}s")
else:
    print("reuse loaded model")

print("model class:", model.__class__)
print("modeling file:", inspect.getfile(model.__class__))
print("inner model class:", model.model.__class__)
print("inner modeling file:", inspect.getfile(model.model.__class__))
print("auto_map:", getattr(model.config, "auto_map", None))
print("use_mla:", model.config.use_mla)
print("_attn_implementation:", getattr(model.config, "_attn_implementation", None))
print("sliding_window:", getattr(model.config, "sliding_window", None))
print("sliding_window_size:", getattr(model.config, "sliding_window_size", None))
print("layer0 attention:", model.model.layers[0].self_attn.__class__)


## 2. 先看 CausalLM forward 的源码

下面这段对应你调用 `model(...)` 之后最先进入的 forward。它本身不处理图片，只把 `images/images_seq_mask/images_spatial_crop` 继续传给 `self.model`。


In [ ]:
show_source(MODELING_UOCR, 595, 691)


## 3. 再看 UnlimitedOCRModel.forward 里图片怎么进入 decoder

这段是最重要的：

- `inputs_embeds = self.get_input_embeddings()(input_ids)` 先拿文本 embedding
- `sam_model + vision_model + projector` 生成视觉 embedding
- `images_seq_mask` 标出 `<image>` token 占位位置
- `masked_scatter_` 把视觉 embedding 塞回 `inputs_embeds`
- 最后 `super(UnlimitedOCRModel, self).forward(...)` 进入 DeepSeekV2 decoder

注意 `with torch.no_grad()` 包住了视觉编码和 projector。按原代码，视觉侧和 projector 默认不会反传。


In [ ]:
show_source(MODELING_UOCR, 449, 592)


## 4. 构造两个独立样本

`build_sample` 复刻了 `infer()` 里的输入构造逻辑，但把 target 文本也拼进去，并生成 labels。这里的 batch=2 是：

```text
sample 0: 一张图 + 一个 prompt + 一个 target
sample 1: 一张图 + 一个 prompt + 一个 target
```

不是 `infer_multi` 那种“一个 conversation 里多张图片”。


In [ ]:
MODE = "gundam"  # "gundam": base_size=1024,image_size=640,crop; "base": 1024 no crop

if MODE == "gundam":
    base_size, image_size, crop_mode = 1024, 640, True
elif MODE == "base":
    base_size, image_size, crop_mode = 1024, 1024, False
else:
    raise ValueError(MODE)

img_a, img_b = ensure_debug_images()
target_a = "BATCH SAMPLE A\nInvoice: A-2026-0001\nDate: 2026-07-07\nTotal 21.25"
target_b = "BATCH SAMPLE B\nInvoice: B-2026-0002\nDate: 2026-07-07\nTotal 34.80"

samples = [
    build_sample(tokenizer, remote_mod, img_a, target_a, base_size=base_size, image_size=image_size, crop_mode=crop_mode),
    build_sample(tokenizer, remote_mod, img_b, target_b, base_size=base_size, image_size=image_size, crop_mode=crop_mode),
]

print("mode:", MODE, "base_size:", base_size, "image_size:", image_size, "crop_mode:", crop_mode)
for i, sample in enumerate(samples):
    print(f"sample {i}:")
    print("  input_ids:", tuple(sample["input_ids"].shape))
    print("  image_tokens:", sample["image_tokens"])
    print("  prompt_len:", sample["prompt_len"])
    print("  supervised tokens:", int((sample["labels"] != -100).sum().item()))
    crop, ori = sample["images"]
    print("  crop/original:", tuple(crop.shape), tuple(ori.shape))
    print("  images_spatial_crop:", sample["images_spatial_crop"])


In [ ]:
# 看一下两个 batch 样本对应的图片。
display(Image.open(img_a))
display(Image.open(img_b))


## 5. Collate 成 batch=2，并检查 forward contract

`UnlimitedOCRModel.forward` 里是：

```python
for image, crop_shape in zip(images, images_spatial_crop):
    ...
    inputs_embeds[idx].masked_scatter_(images_seq_mask[idx].unsqueeze(-1).cuda(), images_in_this_batch)
```

所以这里必须满足：

- `len(images) == batch_size`
- `len(images_spatial_crop) == batch_size`
- `images_seq_mask.shape[0] == batch_size`
- 每个 sample 的 `images_seq_mask.sum()` 要等于该 sample 视觉 embedding 数量


In [ ]:
batch = collate_batch(samples, tokenizer)

print("batch_size:", batch["input_ids"].shape[0])
print("input_ids:", tuple(batch["input_ids"].shape))
print("labels:", tuple(batch["labels"].shape))
print("attention_mask:", tuple(batch["attention_mask"].shape), batch["attention_mask"].sum(dim=1).tolist())
print("images_seq_mask:", tuple(batch["images_seq_mask"].shape), batch["images_seq_mask"].sum(dim=1).tolist())
print("len(images):", len(batch["images"]))
print("image tensor shapes:", [(tuple(crop.shape), tuple(ori.shape)) for crop, ori in batch["images"]])
print("images_spatial_crop:", batch["images_spatial_crop"])
print("prompt_lens:", [s["prompt_len"] for s in samples])
print("target_token_counts:", [(s["labels"] != -100).sum().item() for s in samples])

assert batch["input_ids"].shape[0] == 2
assert len(batch["images"]) == 2
assert len(batch["images_spatial_crop"]) == 2
assert batch["images_seq_mask"].shape[0] == 2
print("batch2 contract ok")


## 6. 真正 forward

在下面 cell 的 `outputs = model(...)` 打断点，然后 Step Into。

这次设置：

```python
use_cache=False
```

所以它是训练/打 loss 风格的完整序列 forward：不创建 `DynamicCache`，也不会进入 ring-buffer decode 的覆盖逻辑。即使第一层 attention 类是 `SlidingWindowLlamaAttention`，它在这个场景下也走普通 attention 路径。


In [ ]:
WITH_LABELS = True
USE_CACHE_FOR_FORWARD = False

images_cuda = [(crop.cuda(), ori.cuda()) for crop, ori in batch["images"]]
labels = batch["labels"].cuda() if WITH_LABELS else None

print("About to call model(...)")
print("  class:", model.__class__.__name__)
print("  use_cache:", USE_CACHE_FOR_FORWARD)
print("  expected first stop: modeling_unlimitedocr.py:617 UnlimitedOCRForCausalLM.forward")

# BREAKPOINT HERE: Step Into this call.
with torch.no_grad():
    with torch.autocast("cuda", dtype=torch.bfloat16):
        outputs = model(
            input_ids=batch["input_ids"].cuda(),
            attention_mask=batch["attention_mask"].cuda(),
            labels=labels,
            images=images_cuda,
            images_seq_mask=batch["images_seq_mask"].cuda(),
            images_spatial_crop=batch["images_spatial_crop"],
            use_cache=USE_CACHE_FOR_FORWARD,
            return_dict=True,
        )

print("logits:", tuple(outputs.logits.shape), outputs.logits.dtype)
if outputs.loss is not None:
    print("loss:", float(outputs.loss.detach().cpu()))
print("past_key_values:", type(outputs.past_key_values), outputs.past_key_values is None)


## 7. 看 labels 里哪些 token 参与 loss

这里可以确认：prompt 部分和 image token 位置都是 `-100`，真正监督的是 assistant target。


In [ ]:
for i, sample in enumerate(samples):
    label_pos = (sample["labels"] != -100).nonzero(as_tuple=True)[0]
    print(f"sample {i}: supervised token count =", len(label_pos))
    print("first supervised pos:", int(label_pos[0]), "last supervised pos:", int(label_pos[-1]))
    print(tokenizer.decode(sample["input_ids"][label_pos].tolist()))
    print("-" * 80)


## 8. sliding-window 逻辑在哪里

普通 batch2 forward 不会真的用 ring-buffer decode。sliding-window 在这里有两层意思：

1. logits 层面的 no-repeat window：`SlidingWindowNoRepeatNgramProcessor`，在 `modeling_unlimitedocr.py:354-383`，用于限制生成重复。
2. KV cache 层面的 ring buffer：`SlidingWindowLlamaAttention`，在 `modeling_deepseekv2.py:1232-1377`，用于长 decode 时限制新增 token 的 KV 增长。

KV ring-buffer 在 `generate()` 时这样接上：

1. `model.infer()` 或 `model.infer_multi()` 设置：

```python
_orig_sw = getattr(self.config, 'sliding_window_size', None) or getattr(self.config, 'sliding_window', None)
self.config._ring_window = _orig_sw
self.config.sliding_window = None
output_ids = self.generate(..., use_cache=True)
```

2. `generate()` 每步都会调用 `prepare_inputs_for_generation()`。
3. `prepare_inputs_for_generation()` 只在 prefill 时传图片，decode 后续 step 不再重复传 `images`。
4. decoder layer 用的是 `SlidingWindowLlamaAttention`。
5. `SlidingWindowLlamaAttention.forward()` 读取 `config._ring_window`。超过窗口后，它在 KV cache 的 ring 区域循环覆盖。


In [ ]:
print("prepare_inputs_for_generation in modeling_unlimitedocr.py")
show_source(MODELING_UOCR, 694, 774)


In [ ]:
print("infer() sets _ring_window before generate")
show_source(MODELING_UOCR, 996, 1045)


In [ ]:
print("decoder layer chooses SlidingWindowLlamaAttention because use_mla=False and _attn_implementation=eager")
show_source(MODELING_DSV2, 1380, 1405)


In [ ]:
print("SlidingWindowLlamaAttention ring-buffer decode path")
show_source(MODELING_DSV2, 1232, 1377)


## 9. 可选：用 batch=2 generate 路径触发 sliding-window 调用链

默认不要跑这一格。你要 debug sliding-window 时再把 `RUN_GENERATE_PATH = True`。

这里不用 `model.infer`，因为 `infer` 是单图单样本路径；也不用 `infer_multi`，因为它是一个 conversation 多图，不是 batch=2。这里直接从前面 `samples` 里截出 prompt-only 部分，构造真正 batch=2 的 `model.generate(...)`。

断点建议：

```text
modeling_unlimitedocr.py:694   prepare_inputs_for_generation
modeling_unlimitedocr.py:710   ring cache 后只取最后 token
modeling_unlimitedocr.py:735   生成 position_ids
modeling_unlimitedocr.py:752   生成 cache_position
modeling_unlimitedocr.py:760   only pass images on prefill
modeling_deepseekv2.py:1282   W = config._ring_window
modeling_deepseekv2.py:1315   true prefill path
modeling_deepseekv2.py:1335   warmup path
modeling_deepseekv2.py:1361   ring overwrite path; 需要生成超过 W 个 token 才容易进
```

`max_new_tokens` 小的时候通常只能看到 prefill/warmup；要看到 overwrite，生成 token 数要超过 `sliding_window=128`，会慢一些。


In [ ]:
RUN_GENERATE_PATH = False
MAX_NEW_TOKENS = 8

if RUN_GENERATE_PATH:
    from torch.nn.utils.rnn import pad_sequence

    prompt_ids = [s["input_ids"][:s["prompt_len"]] for s in samples]
    prompt_image_masks = [s["images_seq_mask"][:s["prompt_len"]] for s in samples]

    pad_id = tokenizer.pad_token_id
    if pad_id is None:
        pad_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else 0

    input_ids = pad_sequence(prompt_ids, batch_first=True, padding_value=pad_id).cuda()
    images_seq_mask = pad_sequence(prompt_image_masks, batch_first=True, padding_value=False).cuda()
    attention_mask = input_ids.ne(pad_id).cuda()
    images = [(s["images"][0].cuda(), s["images"][1].cuda()) for s in samples]
    images_spatial_crop = [s["images_spatial_crop"] for s in samples]

    print("generate batch_size:", input_ids.shape[0])
    print("prompt input_ids:", tuple(input_ids.shape))
    print("prompt image tokens:", images_seq_mask.sum(dim=1).tolist())
    print("len(images):", len(images))

    orig_sw = getattr(model.config, "sliding_window_size", None) or getattr(model.config, "sliding_window", None)
    old_sliding_window = getattr(model.config, "sliding_window", None)
    model.config._ring_window = orig_sw
    model.config.sliding_window = None
    print("_ring_window:", model.config._ring_window)

    try:
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
            generated = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                images=images,
                images_seq_mask=images_seq_mask,
                images_spatial_crop=images_spatial_crop,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                use_cache=True,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=pad_id,
            )
        print("generated shape:", tuple(generated.shape))
        for i in range(generated.shape[0]):
            print(f"sample {i} generated:")
            print(tokenizer.decode(generated[i, input_ids.shape[1]:].tolist(), skip_special_tokens=False))
            print("-" * 80)
    finally:
        model.config.sliding_window = old_sliding_window
else:
    print("Skipped. Set RUN_GENERATE_PATH=True when you want to debug batch=2 generate/sliding-window.")


## 10. 清理显存

只有确认后面不继续复用模型时再跑。


In [ ]:
# del outputs, images_cuda, batch, samples
# del model, tokenizer, remote_mod
# torch.cuda.empty_cache()
